# 📊 LDH Adsorption – Supplementary Data Analysis (T1: interpolation)
**Mục tiêu:** tạo bộ *supplementary data* cho paper *"Interpolation versus generalization in machine learning prediction of
micropollutant adsorption on layered double hydroxides"* — thống kê mô tả, preprocessing, KNN imputation, feature engineering,
VIF, Spearman (bootstrap CI) — và **xuất dataset sau xử lý + danh sách biến đưa vào SHAP** cho **task T1 (random row split)**,
sao cho **tái lập đúng** mô hình chính (`ldh_pipeline.py` + `20260804_ldh_adsorption_analysis.ipynb`).

### Nguyên tắc reproducibility
- Mọi bước *ảnh hưởng tới model input* (load/clean, engineered features, KNN imputation, OrdinalEncoder, split, BorutaShap) **gọi trực tiếp `ldh_pipeline` (`lp.*`)** thay vì copy code → không thể lệch với pipeline chính.
- Split + BorutaShap + preprocessing chạy bằng chính `lp.run_experiment` (chỉ khác `models=("LR",)` để bỏ qua Optuna) → **cùng split, cùng feature selection, cùng SHAP variables**.
- Các cell EDA thuần mô tả (thống kê, Q-Q, outlier, VIF, Spearman) tự viết ở đây, dùng `SEED` cố định.

### Cấu trúc
| # | Nội dung |
|---|---|
| 0 | Import, config (seed, đường dẫn, các run) & load dữ liệu + audit reproducibility |
| 1 | Data analysis: mean / median / std / range / Q1 / Q3 / skewness / Q-Q |
| 2 | Preprocessing: làm sạch categorical, chuẩn hoá `removal_pct` |
| 3 | Missing → engineered features → KNN imputation + encoding → outliers → scaling policy → VIF → Spearman CI |
| 4 | Replica T1 của mô hình chính → dataset sau xử lý & **biến dùng cho SHAP** |
| 5 | Export (CSV / XLSX / manifest) |

## 0. Import, config & load dữ liệu
> Colab: chạy `!pip install Boruta BorutaShap "scipy<1.12.0" cubist optuna shap statsmodels openpyxl` rồi restart runtime
> (BorutaShap cần `scipy<1.12` và `numpy<2`). Đặt `ldh_pipeline.py` cùng thư mục với notebook (như notebook chính).

In [ ]:
# ---- Standard library & third-party ------------------------------------------------------
import contextlib, dataclasses, hashlib, inspect, io, json, math, os, platform, random, sys, warnings
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd()))          # folder that holds ldh_pipeline.py (same as the main notebook)

# ---- Project pipeline: single source of truth for everything that touches model inputs ---
import ldh_pipeline as lp

# ---- Plot style --------------------------------------------------------------------------
plt.rcParams.update({
    'figure.facecolor': '#F8F9FA', 'axes.facecolor': '#FAFAFA',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titleweight': 'bold', 'axes.titlesize': 12, 'font.family': 'DejaVu Sans',
})
BLUE, GREEN, AMBER, RED = '#1976D2', '#66BB6A', '#FFA726', '#FF7043'

def severity_colors(values, warn, bad):
    # Traffic-light colour per value: green <= warn < amber <= bad < red (used for missing %, |skew|, VIF).
    return [RED if v > bad else AMBER if v > warn else GREEN for v in values]

In [ ]:
# ---- Reproducibility config --------------------------------------------------------------
SEED = lp.SEED                   # 42: shared with the main pipeline (split, CV, Optuna, estimators)
random.seed(SEED); np.random.seed(SEED); os.environ['PYTHONHASHSEED'] = str(SEED)
# NOTE: lp.* functions seed themselves explicitly. BorutaShap re-seeds np.random with its own default
# random_state=0 inside fit() (see audit 0.1), so the global seeds above only cover this notebook's own code.

TARGET = 'removal_pct'         # removal efficiency (%), the model target
N_BOOT = 2000                    # bootstrap replicates for the Spearman CIs
SPLIT_MODE = 'row'               # T1 = random row-wise split (interpolation). T2 (group split) is out of scope here.

# T1 runs. `engineered=False` is EXACTLY the main notebook's configuration; `engineered=True` adds
# the chemistry-informed features (dph, c0_per_dose, logkow_x_dph).
RUNS = {
    'main_T1_row': {'engineered': False},
    'ext_T1_row':  {'engineered': True},
}
MAIN_ARTIFACT_FOLDER = 'main_interp_row_42'      # where the main notebook saved its T1 experiment

# ---- Paths -------------------------------------------------------------------------------
FILE_NAME = '20261028_Data_Adsorption.xlsx'
FILE_PATH = lp.DATA_DIR / FILE_NAME
# Colab override:
# FILE_PATH = Path('/content/drive/MyDrive/Green-X Project/Data/20261028_Data_Adsorption.xlsx')

SUPP_DIR = lp.DATA_DIR / 'supplementary'         # all outputs of this notebook
FIG_DIR = SUPP_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
MAIN_MODEL_DIR = lp.MODEL_DIR                    # main notebook's outputs (read-only here)

def save_fig(name):
    plt.savefig(FIG_DIR / name, dpi=300, bbox_inches='tight')

print('Data file :', FILE_PATH)
print('Output dir:', SUPP_DIR)

In [ ]:
# Load dữ liệu
df_raw = pd.read_excel(FILE_PATH, sheet_name=0, header=1)   # header nằm ở row 2
df_raw.columns = df_raw.columns.str.strip()                  # same as lp.load_data

print(f'Shape gốc: {df_raw.shape}')
print(f'Cột: {df_raw.columns.tolist()}')

# Fail fast if the Excel headers are not the standardized names used by ldh_pipeline.py / this notebook.
EXPECTED_COLUMNS = [
    'study_id', 'pollutant', 'log_kow', 'mol_weight',
    'abraham_e', 'abraham_s', 'abraham_a', 'abraham_b', 'abraham_v', 'tpsa',
    'metal_ii', 'metal_iii', 'm2_m3_ratio', 'avg_atomic_weight', 'avg_d_electrons', 'avg_electronegativity',
    'synthesis_method', 'hydrothermal_temp', 'hydrothermal_time', 'calcination_temp', 'calcination_time',
    'd003', 's_bet', 'pore_volume', 'pore_diameter', 'ph_pzc',
    'ph', 'temperature', 'contact_time', 'adsorbent_dose', 'c0', 'removal_pct',
]
missing_cols = [c for c in EXPECTED_COLUMNS if c not in df_raw.columns]
extra_cols = [c for c in df_raw.columns if c not in EXPECTED_COLUMNS]
assert not missing_cols, f'Excel headers are not the standardized names. Missing: {missing_cols} | unrecognised: {extra_cols}'
print('Column names OK:', len(EXPECTED_COLUMNS), 'standardized columns', '| extra:', extra_cols or 'none')

# Fingerprint of the input file, stored in the manifest so a reader can verify they use the same data.
DATA_SHA256 = hashlib.sha256(Path(FILE_PATH).read_bytes()).hexdigest()
print('SHA-256:', DATA_SHA256[:16], '...')
df_raw.head(5)

### 0.1 Audit reproducibility (đọc từ `ldh_pipeline.py` + notebook chính)

| Thành phần | Cấu hình trong pipeline chính | Tác động tới notebook này |
|---|---|---|
| Train/test split (T1) | `train_test_split(test_size=0.2, random_state=SEED)` sau khi bỏ dòng trùng feature (`dedup_features=True`) | Dùng lại `lp.run_experiment` → cùng split |
| CV | `KFold(shuffle=True, random_state=SEED)`; report-CV dùng `SEED+1` | Không dùng ở đây |
| Optuna / estimators | `TPESampler(seed=42)`, mọi estimator `random_state=SEED` | Không cần (không train model cuối) |
| **BorutaShap** | `fit(...)` **không truyền `random_state`** → mặc định `0` (không phải 42). RF bên trong không seed riêng nhưng dùng global `np.random` đã `seed(0)`; chạy đơn luồng | Gọi qua `lp.run_experiment` → giống hệt. Đổi phiên bản `shap`/`BorutaShap`/`sklearn` hoặc `n_trials` có thể đổi kết quả |
| KNN imputer | `ScaledKNNImputer(n_neighbors=5)`, `weights='uniform'` (mặc định), **fit trên train split** | Snippet ban đầu dùng `weights='distance'` → khác pipeline. Mục 3.4 đo mức chênh |
| Engineered features | `ExperimentConfig.engineered_features=False` (mặc định) — **notebook chính không bật** | Chạy 2 variant: `main` (False, = notebook chính) và `extended` (True) |
| Thứ tự bước | `run_experiment` tạo engineered features **trước** khi impute | Notebook theo pipeline; mục 3.4 so sánh với thứ tự "impute → rồi tính feature" |
| Làm sạch categorical | `load_data` strip + lowercase mọi cột text (gồm `study_id`), gộp `synthesis_method` theo `METHOD_MAP` | Xem mục 2.2 |

Nếu thư mục `Model/main_interp_row_42` của notebook chính tồn tại, mục 4.4 đối chiếu trực tiếp với `saved_models.joblib` / `config.json` của nó.

In [ ]:
def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return 'n/a'

VERSIONS = {'python': platform.python_version(),
            **{p: package_version(p) for p in ('numpy', 'pandas', 'scipy', 'scikit-learn', 'shap',
                                               'BorutaShap', 'optuna', 'statsmodels')}}
print('Default ExperimentConfig:', dataclasses.asdict(lp.ExperimentConfig(name='_')))
print('Versions:', VERSIONS)

## 1. Data analysis
Thống kê mô tả trên dữ liệu **gốc (chưa impute)**; `removal_pct` đã đưa về đơn vị % bằng `lp.normalize_removal_pct`.
> Lưu ý: descriptor của pollutant/material lặp lại cho mọi dòng của cùng một study → thống kê theo *dòng* bị trọng số theo số dòng/study.

In [ ]:
def skew_class(skew):
    # Common rule of thumb: |s|<0.5 symmetric, 0.5-1 moderately skewed, >1 highly skewed.
    a = abs(skew)
    return 'symmetric' if a < 0.5 else 'moderate' if a < 1 else 'highly skewed'

def describe_features(data, cols):
    # Per-feature descriptive statistics on OBSERVED values (NaNs dropped per column).
    rows = []
    for c in cols:
        s = data[c].dropna().astype(float)
        q1, q3 = s.quantile([.25, .75])
        constant = s.nunique() <= 1                      # skew/kurtosis/Shapiro are undefined for constants
        rows.append({
            'feature': c, 'n': len(s), 'missing': int(data[c].isna().sum()),
            'mean': s.mean(), 'median': s.median(), 'std': s.std(),
            'min': s.min(), 'Q1': q1, 'Q3': q3, 'max': s.max(), 'range': s.max() - s.min(), 'IQR': q3 - q1,
            'skewness': np.nan if constant else stats.skew(s, bias=False),
            'excess_kurtosis': np.nan if constant else stats.kurtosis(s, bias=False),
            'shapiro_p': np.nan if constant else stats.shapiro(s).pvalue,
        })
    out = pd.DataFrame(rows).set_index('feature')
    out['skew_class'] = out['skewness'].map(lambda v: skew_class(v) if pd.notna(v) else 'n/a')
    out['normal (Shapiro p>0.05)'] = out['shapiro_p'] > 0.05
    return out

# Strip column names + put removal_pct in percent so this table matches what the paper reports.
d1 = df_raw.copy()
d1.columns = d1.columns.str.strip()
d1[TARGET] = lp.normalize_removal_pct(d1[TARGET])
numeric_cols_raw = d1.select_dtypes(include=[np.number]).columns.tolist()

stats_df = describe_features(d1, numeric_cols_raw)
pd.set_option('display.width', 220, 'display.max_columns', 30)
stats_df.round(3)

**Đọc kết quả:** Shapiro–Wilk với n ≈ 10³ cực nhạy (p<0.05 gần như luôn xảy ra), nên quyết định dựa vào **skewness + Q-Q plot**, không chỉ p-value.

In [ ]:
def plot_qq_grid(data, cols, skewness, ncols=6):
    # One normal Q-Q panel per numeric feature; deviation from the dashed line = departure from normality.
    nrows = math.ceil(len(cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.1 * ncols, 2.9 * nrows))
    for ax, c in zip(axes.ravel(), cols):
        s = data[c].dropna().astype(float)
        ax.set_title(c, fontsize=9)
        if s.nunique() <= 1:
            ax.text(.5, .5, 'constant', ha='center'); continue
        (osm, osr), (slope, intercept, r) = stats.probplot(s, dist='norm')
        ax.scatter(osm, osr, s=6, alpha=.5, color=BLUE)
        ax.plot(osm, slope * osm + intercept, 'k--', lw=1)
        ax.set_title(f'{c}\nskew={skewness[c]:.2f}  r={r:.3f}', fontsize=9)
    for ax in axes.ravel()[len(cols):]:                  # hide unused panels
        ax.axis('off')
    fig.suptitle('Normal Q-Q plots (observed values)', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout(); save_fig('fig1_qq_plots.png'); plt.show()

plot_qq_grid(d1, numeric_cols_raw, stats_df['skewness'])

In [ ]:
skew = stats_df['skewness'].dropna().sort_values()
fig, ax = plt.subplots(figsize=(8, 0.28 * len(skew) + 1.5))
ax.barh(skew.index, skew.values, color=severity_colors(skew.abs(), warn=.5, bad=1))
for x in (-1, -.5, .5, 1):
    ax.axvline(x, color='gray', ls=':', lw=1)
ax.set_xlabel('Skewness (|s|<0.5 symmetric, 0.5–1 moderate, >1 high)')
ax.set_title('Skewness of numeric variables')
plt.tight_layout(); save_fig('fig2_skewness.png'); plt.show()
print('Highly skewed (|s|>1):', skew[skew.abs() > 1].index.tolist())

## 2. Preprocessing
Dùng đúng `lp.load_data`: strip tên cột → strip + lowercase mọi cột text → gộp `synthesis_method` theo `METHOD_MAP` → `removal_pct` về [0, 100].
Source thực sự được chạy được in ra để đối chiếu.

In [ ]:
print(json.dumps(lp.METHOD_MAP, indent=2))
print(inspect.getsource(lp.load_data))
print(inspect.getsource(lp.normalize_removal_pct))

In [ ]:
df = lp.load_data(FILE_PATH)                 # == the preprocessing used by the main model
# Row positions must equal original Excel rows: the exported `source_row` relies on this.
assert (df.index == np.arange(len(df))).all() and df[TARGET].notna().all()

print(f'Shape: {df.shape} | studies (study_id): {df["study_id"].nunique()} | '
      f'removal_pct raw max={df_raw[TARGET].max():.3f} -> clean range=[{df[TARGET].min():.2f}, {df[TARGET].max():.2f}]')
# synthesis_method synonyms before -> after the METHOD_MAP merge
pd.crosstab(df_raw['synthesis_method'], df['synthesis_method'],
            rownames=['synthesis_method (raw)'], colnames=['synthesis_method (clean)'])

### 2.2 Data-quality audit: nhãn categorical bị tách đôi
`lp.load_data` strip + lowercase **mọi** cột text, nên các nhãn chỉ khác nhau về hoa/thường hoặc khoảng trắng (ví dụ `Zn` / `zn`, `Ni ` / `Ni`) được gộp trước khi `OrdinalEncoder` mã hoá. Cell dưới đối chiếu dữ liệu thô với dữ liệu đã làm sạch.

In [ ]:
def label_collisions(series):
    # Labels that differ only by case / surrounding whitespace, e.g. 'Ni ' vs 'Ni', 'zn' vs 'Zn'.
    raw = series.astype(str)
    key = raw.str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
    groups = pd.DataFrame({'raw': raw, 'key': key}).groupby('key')['raw'].agg(lambda v: sorted(set(v)))
    return groups[groups.map(len) > 1]

numeric_clean = df.select_dtypes(include=[np.number]).columns
categorical_audit = [c for c in df.columns if c not in numeric_clean and c != 'study_id']
rows = []
for c in categorical_audit:
    before, after = label_collisions(df_raw[c]), label_collisions(df[c])
    rows.append({'feature': c,
                 'n_labels_raw': df_raw[c].astype(str).nunique(), 'n_labels_clean': df[c].nunique(),
                 'collisions_raw': before.to_dict() or 'none',
                 'collisions_after_load_data': after.to_dict() or 'none'})
    assert after.empty, f'{c}: labels still split after lp.load_data'
display(pd.DataFrame(rows).set_index('feature'))
print()
print(df['metal_ii'].map(repr).value_counts().to_string())

> ✅ **Kết quả:** sau `lp.load_data` không còn nhãn bị tách đôi (có `assert` ở cell trên). Với file `20261028_Data_Adsorption.xlsx`, `metal_ii` giảm từ 10 xuống 8 nhãn (`Ni ` → `ni`; `Zn` và `zn` → `zn`).
> ⚠️ **Hệ quả:** (i) mọi nhãn text là **chữ thường** (`zn`, `diclofenac`…); (ii) mã `OrdinalEncoder` và kết quả model **khác** các artifact chạy trước khi đổi pipeline → phải chạy lại notebook chính (`Model/main_interp_row_42`) trước khi đối chiếu ở mục 4.4.

## 3. Missing → feature engineering → KNN imputation → outliers → VIF → correlation
### 3.1 Missing values

In [ ]:
missing_df = pd.DataFrame({'Missing count': df.isnull().sum(),
                           'Missing %': (df.isnull().mean() * 100).round(1)})
missing_df = missing_df[missing_df['Missing count'] > 0].sort_values('Missing %')

if len(missing_df):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(missing_df.index, missing_df['Missing %'], color=severity_colors(missing_df['Missing %'], warn=10, bad=20))
    for y_pos, v in enumerate(missing_df['Missing %']):
        ax.text(v + .5, y_pos, f'{v:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('Missing (%)'); ax.set_title('Missing values by column')
    plt.tight_layout(); save_fig('fig3_missing.png'); plt.show()
else:
    print('No missing values')
missing_df

> `calcination_temp` / `calcination_time` thiếu ≈ 66% — KNN imputation ở mức này chủ yếu "mượn" từ các study có ghi nhận, cần nêu rõ khi viết Methods.

### 3.2 Feature engineering (đúng thứ tự `run_experiment`: tạo feature **trước** khi impute)

In [ ]:
print(inspect.getsource(lp.add_engineered_features))

X_raw = df.drop(columns=['study_id', TARGET])                  # features only: no study id, no target
X_eng = lp.add_engineered_features(X_raw)                # raw features + engineered ones (still with NaN)
ENG_FEATURES = [c for c in X_eng.columns if c not in X_raw.columns]
print('Engineered features:', ENG_FEATURES)
# Guard: if lp.add_engineered_features still looks for the old column names it silently adds nothing.
assert ENG_FEATURES == ['dph', 'c0_per_dose', 'logkow_x_dph'], (
    f'add_engineered_features returned {ENG_FEATURES}: update it to the standardized column names in ldh_pipeline.py')

eng_summary = X_eng[ENG_FEATURES].describe().T
eng_summary['missing'] = X_eng[ENG_FEATURES].isna().sum()
eng_summary.round(3)

### 3.3 KNN imputation (z-score space → trả về đơn vị gốc) + OrdinalEncoder
`lp.make_preprocessor` = `ColumnTransformer[ScaledKNNImputer (numeric), OrdinalEncoder (categorical)]` – chính transformer nằm trong mọi model pipeline.
Ở đây fit trên **toàn bộ dữ liệu** (chỉ để mô tả / VIF / correlation → dataset **S1**); trong model thật nó fit trên *train split* (mục 4).

In [ ]:
print(inspect.getsource(lp.ScaledKNNImputer))

num_cols, cat_cols = lp.split_feature_types(X_eng)
BASE_NUM = [c for c in num_cols if c not in ENG_FEATURES]           # numeric features before engineering

prep_full = lp.make_preprocessor(num_cols, cat_cols)
S1_feat = prep_full.fit_transform(X_eng)                            # imputed (original units) + ordinal-encoded
assert S1_feat.isna().sum().sum() == 0 and list(S1_feat.index) == list(X_eng.index)
print('S1 feature matrix:', S1_feat.shape, '| numeric first, then categorical:', list(S1_feat.columns)[-len(cat_cols):])

In [ ]:
# Sanity check on the imputation: imputed cells should look like the observed distribution (no large shift).
was_missing = X_eng[num_cols].isna()
imputation_check = pd.DataFrame({
    'n_imputed': was_missing.sum(),
    'pct_imputed': (was_missing.mean() * 100).round(1),
    'mean_observed': [X_eng.loc[~was_missing[c], c].mean() for c in num_cols],
    'mean_imputed_cells': [S1_feat.loc[was_missing[c], c].mean() if was_missing[c].any() else np.nan for c in num_cols],
    'std_observed': [X_eng.loc[~was_missing[c], c].std() for c in num_cols],
})
imputation_check[imputation_check.n_imputed > 0].round(3)

In [ ]:
# Category -> integer code learned by OrdinalEncoder (needed to interpret categorical SHAP values).
encoder = prep_full.named_transformers_['cat']
enc_maps = pd.DataFrame([{'feature': c, 'category': cat, 'code': i}
                         for c, cats in zip(cat_cols, encoder.categories_) for i, cat in enumerate(cats)])
display(enc_maps.groupby('feature').agg(n_categories=('code', 'size')))
enc_maps

### 3.4 Diagnostics: hai điểm pipeline khác mô tả ban đầu
**(a) `weights='distance'` vs `'uniform'`** – pipeline dùng `uniform` (mặc định `KNNImputer`). Đo chênh lệch trên các ô thực sự được impute (đơn vị z-score).

In [ ]:
Z = StandardScaler().fit_transform(X_raw[BASE_NUM].to_numpy(float))          # NaN-aware scaling
impute = lambda **kw: KNNImputer(n_neighbors=5, keep_empty_features=True, **kw).fit_transform(Z)
delta = np.abs(impute() - impute(weights='distance'))                        # |uniform - distance| in z units
imputed_cells = np.isnan(Z)                                                  # only compare cells that were imputed
weights_diff = pd.Series({c: delta[imputed_cells[:, j], j].mean()
                          for j, c in enumerate(BASE_NUM) if imputed_cells[:, j].any()},
                         name='mean |Δz| (uniform vs distance)')
weights_diff.round(3).to_frame()

**(b) Thứ tự "impute → tính feature" vs "tính feature → impute"** – pipeline tính `dph` (= pH − pH_pzc) từ giá trị thô rồi KNN-impute chính feature đó; thứ tự ban đầu sẽ tính từ `ph_pzc` đã impute.

In [ ]:
ph_pzc_missing = X_eng['ph_pzc'].isna()
engineer_then_impute = S1_feat['dph']                                         # what the pipeline does
impute_then_engineer = S1_feat['ph'] - S1_feat['ph_pzc']                      # the alternative order
diff = (impute_then_engineer - engineer_then_impute)[ph_pzc_missing]          # only rows where ph_pzc was imputed
print(f'rows with missing ph_pzc: {ph_pzc_missing.sum()} ({ph_pzc_missing.mean() * 100:.1f}%)')
print(f'dph, two orders: mean|diff|={diff.abs().mean():.3f}, max|diff|={diff.abs().max():.3f}, '
      f'corr={np.corrcoef(impute_then_engineer[ph_pzc_missing], engineer_then_impute[ph_pzc_missing])[0, 1]:.3f}')

### 3.5 Outliers
Phát hiện trên **giá trị quan sát được** (trước impute) bằng quy tắc IQR (1.5×IQR = mild, 3×IQR = extreme).
**Xử lý: giữ lại, chỉ đánh dấu** — lý do: (i) đây là số đo thực nghiệm (`s_bet`, `c0`, `contact_time`… trải nhiều bậc độ lớn); (ii) mô hình cây bất biến với phép biến đổi đơn điệu nên ít nhạy outlier;
(iii) descriptor lặp theo study nên "outlier" thường là *cả một study* – loại bỏ sẽ mất nguyên một hệ vật liệu/pollutant; (iv) Spearman dựa trên hạng nên bền vững.

In [ ]:
def iqr_outlier_screen(X, cols, study_ids):
    # Tukey fences on observed values. Returns a summary table and a boolean (row x feature) mild-outlier mask.
    rows, mild_mask = [], pd.DataFrame(False, index=X.index, columns=cols)
    for c in cols:
        s = X[c]; obs = s.dropna()
        q1, q3 = obs.quantile([.25, .75]); iqr = q3 - q1
        mild = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)      # NaN compares False -> never flagged
        extreme = (s < q1 - 3 * iqr) | (s > q3 + 3 * iqr)
        mild_mask[c] = mild
        rows.append({'feature': c, 'n_obs': len(obs), 'IQR': iqr,
                     'lower_fence': q1 - 1.5 * iqr, 'upper_fence': q3 + 1.5 * iqr,
                     'n_mild': int(mild.sum()), 'n_extreme': int(extreme.sum()),
                     'pct_mild': round(mild.sum() / len(obs) * 100, 1),
                     'n_studies_with_mild': study_ids[mild].nunique(),      # outliers cluster by study
                     'skewness': obs.skew(),
                     'note': 'IQR=0: fences degenerate' if iqr == 0 else ''})
    return pd.DataFrame(rows).set_index('feature'), mild_mask

outlier_df, mild_outliers = iqr_outlier_screen(X_raw, BASE_NUM, df['study_id'])
display(outlier_df.round(3))

# Boxplot on a robust scale so features with very different units share one axis.
iqr = X_raw[BASE_NUM].quantile(.75) - X_raw[BASE_NUM].quantile(.25)
scale = iqr.where(iqr > 0, X_raw[BASE_NUM].std())                # IQR=0 (mostly constant) -> fall back to std
robust_z = (X_raw[BASE_NUM] - X_raw[BASE_NUM].median()) / scale

fig, ax = plt.subplots(figsize=(9, 0.32 * len(BASE_NUM) + 1.5))
ax.boxplot([robust_z[c].dropna() for c in BASE_NUM], vert=False, whis=1.5,
           flierprops=dict(marker='.', markersize=3, alpha=.4, markeredgecolor=RED),
           boxprops=dict(color=BLUE), medianprops=dict(color='black'))
ax.set_yticks(range(1, len(BASE_NUM) + 1)); ax.set_yticklabels(BASE_NUM); ax.invert_yaxis()
ax.set_xlim(-8, 12); ax.set_xlabel('robust z = (x − median) / IQR  (std if IQR=0); axis clipped to [−8, 12]')
ax.set_title('Outlier screen (IQR rule), observed values')
plt.tight_layout(); save_fig('fig4_outliers.png'); plt.show()

### 3.6 Chính sách scaling
Chỉ **LR, KNN, ANN, Cubist** được thêm `StandardScaler` (xem `lp.build_models`: `with_scaler=True`); **RF, XGB, LGB, HistGB không scale**.
`ScaledKNNImputer` trả dữ liệu về **đơn vị gốc**, nên dataset xuất ra ở đây **không bị scale** – scaler nằm trong từng model pipeline và fit trên train fold.
(Hằng số `lp.NEEDS_SCALER` hiện không được tham chiếu trong code; việc scale do cờ `with_scaler` quyết định.)

In [ ]:
print('lp.NEEDS_SCALER =', sorted(lp.NEEDS_SCALER))
pd.DataFrame({'model': ['LR', 'KNN', 'ANN', 'Cubist', 'RF', 'XGB', 'LGB', 'HistGB'],
              'StandardScaler step': ['yes'] * 4 + ['no'] * 4}).set_index('model').T

### 3.7 Multicollinearity (VIF)
Tính trên dữ liệu đã impute (`lp.vif_table` bỏ dòng NaN nên không chạy được trên dữ liệu thô: `calcination_*` thiếu ≈66%), chỉ trên **biến số** (không gồm target, không gồm categorical mã hoá thứ tự).
So sánh tập `base` và `extended` (thêm engineered features – vốn cộng tuyến theo cấu trúc với `ph`, `ph_pzc`, `c0`, `adsorbent_dose`).

In [ ]:
S1_numeric = S1_feat[num_cols]
vif_cmp = (lp.vif_table(S1_numeric).rename(columns={'VIF': 'VIF_extended'})
           .merge(lp.vif_table(S1_numeric[BASE_NUM]).rename(columns={'VIF': 'VIF_base'}), on='feature', how='left'))
vif_cmp['engineered'] = vif_cmp['feature'].isin(ENG_FEATURES)
display(vif_cmp.round(2))

# Plot (log axis; inf from perfect collinearity is clipped for display only)
plot_df = vif_cmp.replace(np.inf, 1e6).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 0.28 * len(plot_df) + 1.5))
ax.barh(plot_df['feature'], plot_df['VIF_extended'], color=severity_colors(plot_df['VIF_extended'], warn=5, bad=10))
ax.set_xscale('log'); ax.axvline(5, color='gray', ls=':'); ax.axvline(10, color='gray', ls='--')
ax.set_xlabel('VIF (log scale; dotted=5, dashed=10)'); ax.set_title('Variance Inflation Factor – extended feature set')
plt.tight_layout(); save_fig('fig5_vif.png'); plt.show()
print('VIF > 10 (extended):', vif_cmp.loc[vif_cmp.VIF_extended > 10, 'feature'].tolist())

### 3.8 Spearman rank correlation với `removal_pct` + bootstrap 95% CI
- Tương quan **pairwise-complete trên giá trị quan sát** (không dùng giá trị impute, để không tạo cấu trúc giả).
- **Row bootstrap** (resample dòng) là CI phù hợp với T1 (random row split, các dòng được coi như độc lập).
- **Study-cluster bootstrap** (resample nguyên study theo `study_id`) cho thấy CI theo dòng *lạc quan* tới mức nào, vì các dòng cùng study gần như không độc lập (descriptor lặp lại) —
  đây là lý do T1 có thể đánh giá quá cao khả năng dự đoán.

In [ ]:
def spearman_rho(x, y):
    # Spearman = Pearson on ranks. Undefined for <10 pairs or a constant feature.
    if len(x) < 10 or np.ptp(x) == 0:
        return np.nan
    return np.corrcoef(stats.rankdata(x), stats.rankdata(y))[0, 1]

def spearman_bootstrap(values, y, study_ids, n_boot, seed):
    # values: (n_rows x n_features) with NaN for missing; correlations use pairwise-complete observations.
    # Returns point estimates + 95% percentile CIs for (a) row bootstrap and (b) study-cluster bootstrap.
    observed = ~np.isnan(values)

    def rho_all(idx):                                     # rho for every feature on the rows in `idx`
        return np.array([spearman_rho(values[idx[observed[idx, j]], j], y[idx[observed[idx, j]]])
                         for j in range(values.shape[1])])

    rng = np.random.default_rng(seed)
    studies = np.unique(study_ids)
    rows_of_study = [np.flatnonzero(study_ids == s) for s in studies]
    n = len(y)
    boot_row, boot_cluster = np.empty((n_boot, values.shape[1])), np.empty((n_boot, values.shape[1]))
    for b in range(n_boot):
        boot_row[b] = rho_all(rng.integers(0, n, n))                                   # resample rows
        picked = rng.integers(0, len(studies), len(studies))                            # resample whole studies
        boot_cluster[b] = rho_all(np.concatenate([rows_of_study[k] for k in picked]))

    lo_r, hi_r = np.nanpercentile(boot_row, [2.5, 97.5], axis=0)
    lo_c, hi_c = np.nanpercentile(boot_cluster, [2.5, 97.5], axis=0)
    return pd.DataFrame({'n_pairs': observed.sum(0), 'rho': rho_all(np.arange(n)),
                         'CI95_row_low': lo_r, 'CI95_row_high': hi_r,
                         'CI95_cluster_low': lo_c, 'CI95_cluster_high': hi_c})

corr_feats = BASE_NUM + ENG_FEATURES
spearman_df = spearman_bootstrap(X_eng[corr_feats].to_numpy(float), df[TARGET].to_numpy(float),
                                 df['study_id'].to_numpy(), N_BOOT, SEED).set_index(pd.Index(corr_feats))
excludes_zero = lambda lo, hi: (lo > 0) | (hi < 0)        # CI excludes 0 -> "significant" at 95%
spearman_df['sig_row'] = excludes_zero(spearman_df.CI95_row_low, spearman_df.CI95_row_high)
spearman_df['sig_cluster'] = excludes_zero(spearman_df.CI95_cluster_low, spearman_df.CI95_cluster_high)
spearman_df['engineered'] = spearman_df.index.isin(ENG_FEATURES)
spearman_df = spearman_df.reindex(spearman_df.rho.abs().sort_values(ascending=False).index)
spearman_df.round(3)

In [ ]:
# Forest plot: dot = rho, thick pale bar = row-bootstrap CI, thin dark bar = study-cluster CI.
p = spearman_df.iloc[::-1]
y_pos = np.arange(len(p))
fig, ax = plt.subplots(figsize=(9, 0.34 * len(p) + 1.6))
ax.hlines(y_pos + .12, p.CI95_row_low, p.CI95_row_high, color='#90CAF9', lw=4, label='95% CI – row bootstrap')
ax.hlines(y_pos - .12, p.CI95_cluster_low, p.CI95_cluster_high, color='#1565C0', lw=2, label='95% CI – study-cluster bootstrap')
ax.scatter(p.rho, y_pos, s=28, zorder=3, edgecolor='#1565C0', color=['#1565C0' if s else 'white' for s in p.sig_cluster])
ax.axvline(0, color='black', lw=1)
ax.set_yticks(y_pos); ax.set_yticklabels([f + ' *' if e else f for f, e in zip(p.index, p.engineered)])
ax.set_xlabel(f'Spearman ρ with {TARGET}   (filled = cluster CI excludes 0; * = engineered)')
ax.set_title('Spearman correlation with removal efficiency and bootstrap 95% CI'); ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); save_fig('fig6_spearman_ci.png'); plt.show()
print('Significant by row CI but NOT by study-cluster CI:', spearman_df.index[spearman_df.sig_row & ~spearman_df.sig_cluster].tolist())

## 4. Replica T1 của mô hình chính → dataset sau xử lý & biến dùng cho SHAP
Mỗi run gọi **`lp.run_experiment`** với đúng config của notebook chính (`split_mode='row'`, `dedup_features=True`, `seed=42`, `test_size=0.2`, `target='removal_pct'`),
chỉ đổi `models=("LR",)` để bỏ qua tuning Optuna (split, BorutaShap và preprocessor **không phụ thuộc** vào model). Output phụ ghi vào thư mục riêng, **không đụng `Model/` của notebook chính**.

| key | engineered features | ghi chú |
|---|---|---|
| `main_T1_row` | ❌ | **giống notebook chính** |
| `ext_T1_row` | ✅ | thêm `dph`, `c0_per_dose`, `logkow_x_dph` |

> ⏱ BorutaShap ≈ 3–5 phút/run (đơn luồng) → khoảng 8–10 phút cho 2 run.

In [ ]:
REPLICA_DIR = SUPP_DIR / '_replica_runs'

def run_t1_replica(key, engineered):
    # Run the main pipeline's T1 experiment (split -> BorutaShap -> preprocessing) and return the result + log lines.
    cfg = lp.ExperimentConfig(name=f'supp_{key}', split_mode=SPLIT_MODE, dedup_features=True,
                              engineered_features=engineered, models=('LR',),
                              target=TARGET, data_file=FILE_NAME)   # other fields = main notebook defaults
    stdout, stderr = io.StringIO(), io.StringIO()
    lp.MODEL_DIR = REPLICA_DIR                        # keep the replica's side outputs away from the main Model/ folder
    try:
        with contextlib.redirect_stdout(stdout), contextlib.redirect_stderr(stderr):   # silence prints + tqdm bars
            run = lp.run_experiment(cfg, df)
    finally:
        lp.MODEL_DIR = MAIN_MODEL_DIR
    log = [l for l in stdout.getvalue().splitlines() if l.startswith(('[supp_', '[BorutaShap]'))]
    return run, log

# Fail fast (before the 3-5 min BorutaShap runs) if the pipeline would keep the target inside X.
assert isinstance(lp.TARGET_CANDIDATES, (list, tuple)) and TARGET in lp.TARGET_CANDIDATES, (
    f'lp.TARGET_CANDIDATES={lp.TARGET_CANDIDATES!r} must be a list containing {TARGET!r}')

runs = {}
for key, opts in RUNS.items():
    print(f'>>> {key}: running (BorutaShap ~3-5 min) ...', flush=True)
    runs[key], log = run_t1_replica(key, **opts)
    print(*log, sep='\n  ')

### 4.1 Dataset sau xử lý (cái model nhìn thấy) + biến dùng cho SHAP

In [ ]:
def collect_run(key, run, engineered):
    # Turn one lp.run_experiment result into: model-input table, SHAP variable list, BorutaShap bookkeeping.
    prep = run['results']['LR']['model'].named_steps['prep']        # ColumnTransformer fitted on the TRAIN split only
    X_tr, X_te = run['X_train'], run['X_test']                      # already reduced by BorutaShap
    T_tr, T_te = prep.transform(X_tr), prep.transform(X_te)         # imputed (original units) + ordinal-encoded
    assert list(T_tr.columns) == list(T_te.columns)

    def with_meta(T, split):
        # Prepend traceability columns; index = original row positions (guaranteed by the assert in section 2).
        idx = T.index.to_numpy()
        meta = pd.DataFrame({'source_row': idx + 3,                 # Excel row: header on row 2 -> data starts on row 3
                             'study_id': df.loc[idx, 'study_id'].to_numpy(), 'split': split,
                             TARGET: df.loc[idx, TARGET].to_numpy()}, index=T.index)
        return pd.concat([meta, T], axis=1)

    categorical = set(prep.transformers_[1][2])
    candidates = list(X_raw.columns) + (ENG_FEATURES if engineered else [])
    kind = lambda c: 'engineered' if c in ENG_FEATURES else 'categorical' if c in categorical else 'numeric'
    return {
        'engineered': engineered,
        'data': pd.concat([with_meta(T_tr, 'train'), with_meta(T_te, 'test')]),
        'X_train': X_tr, 'X_test': X_te, 'y_train': run['y_train'],
        'shap_vars': list(T_te.columns),                            # exact columns SHAP receives, in model order
        'dropped': [c for c in candidates if c not in X_tr.columns],
        'candidates': candidates, 'kind': {c: kind(c) for c in candidates},
        'n_train': len(X_tr), 'n_test': len(X_te), 'n_dup_removed': len(df) - len(X_tr) - len(X_te),
    }

res = {k: collect_run(k, runs[k], RUNS[k]['engineered']) for k in RUNS}
for k, r in res.items():
    print(f"\n===== {k}: train={r['n_train']}  test={r['n_test']}  duplicates removed={r['n_dup_removed']} =====")
    print(f"BorutaShap dropped ({len(r['dropped'])}): {r['dropped']}")
    print(f"Variables used for SHAP ({len(r['shap_vars'])}, in model order):")
    for i, c in enumerate(r['shap_vars'], 1):
        print(f"  {i:2d}. {c:36s} [{r['kind'][c]}]")

In [ ]:
# Presence matrix: which candidate features reach SHAP in each run.
all_candidates = list(dict.fromkeys(c for r in res.values() for c in r['candidates']))
symbol = lambda r, c: '✓' if c in r['shap_vars'] else '·' if c in r['candidates'] else '–'
presence = pd.DataFrame({k: [symbol(r, c) for c in all_candidates] for k, r in res.items()}, index=all_candidates)
presence.insert(0, 'kind', ['engineered' if c in ENG_FEATURES else 'categorical' if c in cat_cols else 'numeric'
                            for c in all_candidates])
print('✓ = used for SHAP | · = dropped by BorutaShap | – = not a candidate in that run')
presence

### 4.2 Kiểm tra: tên biến đưa vào SHAP = danh sách ở trên
Fit một pipeline cây nhanh trên đúng `X_train`, rồi gọi `lp.shap_summary` (hàm notebook chính dùng) và so tên biến / số cột với `shap_vars`.

In [ ]:
for k, r in res.items():
    num_f, cat_f = lp.split_feature_types(r['X_train'])
    probe = lp.make_model_pipeline(lp.make_preprocessor(num_f, cat_f),
                                   RandomForestRegressor(n_estimators=50, random_state=SEED, n_jobs=1))
    probe.fit(r['X_train'], r['y_train'])
    shap_values, names = lp.shap_summary({'model': probe}, r['X_test'])      # same call as the main notebook
    assert names == r['shap_vars'] and shap_values.values.shape == (len(r['X_test']), len(names)), k
    print(f'{k:12s} SHAP input OK: {len(names)} variables, shap values {shap_values.values.shape}')

### 4.3 Dataset đầy đủ **S1** (impute trên toàn bộ dữ liệu – chỉ để mô tả)
`is_duplicate_feature_row` = dòng trùng hệt các dòng trước về mọi feature (bị loại trong các run T1 vì `dedup_features=True`);
`n_mild_outlier_features` = số biến base bị đánh dấu IQR-mild (mục 3.5).

In [ ]:
S1_meta = pd.DataFrame({'source_row': df.index + 3, 'study_id': df['study_id'].to_numpy(),
                        'is_duplicate_feature_row': X_raw.duplicated(keep='first').to_numpy(),
                        'n_mild_outlier_features': mild_outliers.sum(axis=1).to_numpy(),
                        TARGET: df[TARGET].to_numpy()}, index=df.index)
S1 = pd.concat([S1_meta, S1_feat], axis=1)
print(S1.shape, '| NaN cells in features:', int(S1_feat.isna().sum().sum()))
S1.head()

### 4.4 Đối chiếu với artifact của notebook chính (nếu có)
So tên biến đầu ra của `prep` và config đã lưu trong `Model/main_interp_row_42` với replica `main_T1_row`.

In [ ]:
import joblib

def crosscheck_with_main(replica_key, folder):
    # Compare the replica with the main notebook's saved T1 run. Returns 'MATCH' / 'MISMATCH' / 'skipped'.
    path = MAIN_MODEL_DIR / folder
    if not (path / 'saved_models.joblib').exists():
        print(f'[skip] {path} not found - main artifacts unavailable, cross-check not run')
        return 'skipped'
    saved_models = joblib.load(path / 'saved_models.joblib')
    saved_vars = list(next(iter(saved_models.values())).named_steps['prep'].get_feature_names_out())
    saved_cfg = json.loads((path / 'config.json').read_text())
    mine = dataclasses.asdict(runs[replica_key]['config'])
    keys = ('seed', 'test_size', 'split_mode', 'dedup_features', 'engineered_features', 'target', 'drop_features')
    cfg_diff = {k: (saved_cfg.get(k), mine[k]) for k in keys if saved_cfg.get(k) != mine[k]}
    same_vars = saved_vars == res[replica_key]['shap_vars']
    status = 'MATCH' if same_vars and not cfg_diff else 'MISMATCH'
    print(f'{replica_key}: SHAP variables {"identical" if same_vars else "DIFFER"} | config diff: {cfg_diff or "none"} -> {status}')
    if not same_vars:
        print('  only in main   :', sorted(set(saved_vars) - set(res[replica_key]['shap_vars'])))
        print('  only in replica:', sorted(set(res[replica_key]['shap_vars']) - set(saved_vars)))
    return status

crosscheck = {'main_T1_row': crosscheck_with_main('main_T1_row', MAIN_ARTIFACT_FOLDER)}

## 5. Export

In [ ]:
def export_all():
    # Write datasets (CSV), supplementary tables (XLSX) and the reproducibility manifest (JSON). Returns file names.
    files = []
    S1.to_csv(SUPP_DIR / 'S1_full_imputed_extended.csv', index=False); files.append('S1_full_imputed_extended.csv')
    for k, r in res.items():
        name = f'S2_modelinput_{k}.csv'
        r['data'].to_csv(SUPP_DIR / name, index=False); files.append(name)

    shap_long = pd.DataFrame([{'run': k, 'position': i, 'variable': c, 'kind': r['kind'][c]}
                              for k, r in res.items() for i, c in enumerate(r['shap_vars'], 1)])
    sheets = {'1_descriptive_stats': (stats_df.round(4), True), '3_missing': (missing_df, True),
              '3_outliers_IQR': (outlier_df.round(4), True), '3_VIF': (vif_cmp, False),
              '3_spearman_bootstrap': (spearman_df.round(4), True), '4_shap_variables': (shap_long, False),
              '4_feature_presence': (presence, True), '4_encoder_maps_full': (enc_maps, False)}
    with pd.ExcelWriter(SUPP_DIR / 'supplementary_tables.xlsx') as xw:
        for sheet, (table, keep_index) in sheets.items():
            table.to_excel(xw, sheet_name=sheet, index=keep_index)
    files.append('supplementary_tables.xlsx')

    manifest = {
        'data_file': FILE_NAME, 'data_sha256': DATA_SHA256, 'n_rows': len(df), 'n_studies': int(df['study_id'].nunique()),
        'SEED': SEED, 'split_mode': SPLIT_MODE, 'n_bootstrap': N_BOOT, 'versions': VERSIONS,
        'boruta_shap': {'random_state': 0, 'n_trials': 20,
                        'note': 'lp.run_borutashap does not pass random_state; BorutaShap default 0 is used'},
        'knn_imputer': {'n_neighbors': 5, 'weights': 'uniform',
                        'fit_on': 'train split (S2) / all rows (S1, descriptive only)'},
        'engineered_features': ENG_FEATURES, 'main_artifact_crosscheck': crosscheck,
        'runs': {k: {'engineered_features': r['engineered'], 'config': dataclasses.asdict(runs[k]['config']),
                     'n_train': r['n_train'], 'n_test': r['n_test'], 'n_duplicates_removed': r['n_dup_removed'],
                     'boruta_dropped': r['dropped'], 'shap_variables': r['shap_vars']} for k, r in res.items()},
    }
    (SUPP_DIR / 'reproducibility_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
    files.append('reproducibility_manifest.json')
    return files

print('Saved to', SUPP_DIR); print(*export_all(), sep='\n  ')

## 6. Ghi chú khi dùng trong paper
- **S2 (model input)** là ma trận đúng như model nhìn thấy: imputer/encoder fit trên *train split*, dòng test dùng tham số của train. Cột `split` cho biết train/test; `source_row` là số dòng trong Excel gốc.
  Category chưa từng xuất hiện trong train được mã hoá `-1`.
- **S1** impute trên toàn bộ dữ liệu → chỉ dùng để mô tả (VIF, Spearman…), **không** dùng để huấn luyện/đánh giá.
- Biến dùng cho SHAP = các cột của S2 (sau BorutaShap), theo thứ tự `numeric → categorical` của `ColumnTransformer`.
- Chỉ T1 (random row split, `dedup_features=True`) được tái lập ở đây; T2 (group split) không nằm trong phạm vi notebook này.